# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdulm111/ML-Assignement01/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item's daily search performance record: a
(`report_date`, `client_hash_id`, `content_hash_id`) triple, from
`fact_content_daily_performance`. Working on the mid-panel partition
`month=2026-03` — never the `_sample` file, which is the final month
(June 2026) and is treated as a sealed test window. Row shape confirmed below.

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"

# Scan once, query many times — avoids repeated HF reads / rate limits
con.execute(f"""
    CREATE OR REPLACE TABLE march_facts AS
    SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')
""")

con.sql("SELECT report_date, client_hash_id, content_hash_id, gsc_avg_position, gsc_impressions FROM march_facts LIMIT 5").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┬───────────────────┬─────────────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ gsc_avg_position  │ gsc_impressions │
│    date     │         varchar         │         varchar          │      double       │      int64      │
├─────────────┼─────────────────────────┼──────────────────────────┼───────────────────┼─────────────────┤
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_b7e512995f79d5a6 │              3.35 │              20 │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_05597932fe4da067 │               0.0 │               1 │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_7a105f548d9c6916 │             4.928 │             125 │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_905aa32a0230694e │               4.0 │               7 │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_a3ea9792f793ec72 │ 2.272727272727273 │              11 │
└─────────────┴──────────────────────

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- **Feature:** `gsc_avg_position`, `gsc_impressions`, `ga4_engaged_sessions`,
  `sessions_ai`, `scroll_events`  each observed independently of whether the
  page got clicked, so each is knowable at the decision moment.
- **Label / proxy:** CTR, computed as `gsc_clicks / gsc_impressions`.
  `gsc_clicks` is therefore never a feature  it's the label's own numerator.
- **Context:** `client_hash_id`, `content_hash_id`, `report_date`, `month`
  salted hash keys, for grouping/joining only, never for the model to learn from.
- **Excluded:** all `ga4_*` and `ai_*` columns whenever `ga4_data_available IS NOT
  TRUE`  unfiltered, they'd mix real zero-engagement with genuinely unknown,
  quietly biasing any average. `gsc_sum_position` excluded too it's the raw
  component `gsc_avg_position` is already derived from, so keeping both would
  double-count the same signal.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql("""
    SELECT gsc_avg_position, gsc_impressions, ga4_engaged_sessions, sessions_ai,
           scroll_events, gsc_clicks, ga4_data_available
    FROM march_facts LIMIT 5
""").show()

┌───────────────────┬─────────────────┬──────────────────────┬─────────────┬───────────────┬────────────┬────────────────────┐
│ gsc_avg_position  │ gsc_impressions │ ga4_engaged_sessions │ sessions_ai │ scroll_events │ gsc_clicks │ ga4_data_available │
│      double       │      int64      │        int64         │    int64    │     int64     │   int64    │      boolean       │
├───────────────────┼─────────────────┼──────────────────────┼─────────────┼───────────────┼────────────┼────────────────────┤
│              3.35 │              20 │                 NULL │        NULL │          NULL │          0 │ NULL               │
│               0.0 │               1 │                 NULL │        NULL │          NULL │          0 │ NULL               │
│             4.928 │             125 │                 NULL │        NULL │          NULL │          1 │ NULL               │
│               4.0 │               7 │                 NULL │        NULL │          NULL │          0 │ NULL 

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Query 1 — grain.** Empty result = the (date, client, content) grain holds.

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql("""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
    FROM march_facts
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1 LIMIT 5
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │   c   │
│    date     │    varchar     │     varchar     │ int64 │
├─────────────┴────────────────┴─────────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘



**Query 2 — row count and date span** for this slice.

In [29]:
con.sql("SELECT COUNT(*) AS n_rows, MIN(report_date) AS first_date, MAX(report_date) AS last_date FROM march_facts").show()

┌─────────┬────────────┬────────────┐
│ n_rows  │ first_date │ last_date  │
│  int64  │    date    │    date    │
├─────────┼────────────┼────────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │
└─────────┴────────────┴────────────┘



**Query 3 — availability.** `IS TRUE` correctly excludes NULL, not just FALSE.

In [30]:
con.sql("""
    SELECT COUNT(*) AS total,
           SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM march_facts
""").show()

┌─────────┬────────────────────┬────────────────────┐
│  total  │ gsc_available_rows │ ga4_available_rows │
│  int64  │       int128       │       int128       │
├─────────┼────────────────────┼────────────────────┤
│ 9841378 │            3611061 │             413966 │
└─────────┴────────────────────┴────────────────────┘



Filtered to rows with real GSC and GA4 data and a minimum impression floor
(one impression makes `gsc_avg_position` noise, not signal). Each is knowable
at the decision moment:

- `gsc_avg_position` — measured independently of any click, at report time.
- `gsc_impressions` — exposure volume, known before any click happens.
- `ga4_engaged_sessions` — same-day on-page behavior, not the search click itself.
- `sessions_ai` — a separate traffic channel, known same-day.
- `scroll_events` — same-day engagement signal, independent of CTR.

In [31]:
feat = con.sql("""
    SELECT
        client_hash_id, content_hash_id, report_date,
        gsc_avg_position,       -- available when: measured at report time, independent of any click
        gsc_impressions,        -- available when: exposure is known before any click happens
        ga4_engaged_sessions,   -- available when: same-day on-page behavior, not the click itself
        sessions_ai,            -- available when: separate traffic channel, known same-day
        scroll_events,          -- available when: same-day engagement signal, independent of CTR
        gsc_clicks,              -- kept ONLY for the trap step next — not a real feature
        CAST(gsc_clicks AS DOUBLE) / NULLIF(gsc_impressions, 0) AS ctr
    FROM march_facts
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions >= 10
      AND ga4_data_available IS TRUE
""").df()
print(f"Rows with GSC and GA4 both legitimately available: {len(feat)}")
feat.head(10)

Rows with GSC and GA4 both legitimately available: 327836


,client_hash_id,content_hash_id,report_date,gsc_avg_position,gsc_impressions,ga4_engaged_sessions,sessions_ai,scroll_events,gsc_clicks,ctr
0,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,2026-03-01,5.666667,39,0,1,0,0,0.000000
1,client_65de48885f4ef01b,content_e25ea7297a1dffd3,2026-03-01,5.156425,179,0,1,0,0,0.000000
2,client_65de48885f4ef01b,content_6b0149a80607dac3,2026-03-01,7.694444,72,0,1,0,0,0.000000
3,client_65de48885f4ef01b,content_62673eea26c31c17,2026-03-01,6.167885,3282,0,0,0,1,0.000305
4,client_65de48885f4ef01b,content_872342e050545a12,2026-03-01,6.538462,39,0,0,0,0,0.000000
5,client_65de48885f4ef01b,content_3c286ded8bd68120,2026-03-01,8.431818,88,0,0,0,1,0.011364
6,client_65de48885f4ef01b,content_b2108e8fe3360fa6,2026-03-01,5.300000,40,0,0,0,1,0.025000
7,client_65de48885f4ef01b,content_4c185d1c173cd53d,2026-03-01,30.304348,23,0,0,0,0,0.000000
8,client_65de48885f4ef01b,content_bd07be40ea0d5f54,2026-03-01,5.478261,23,0,0,0,0,0.000000
9,client_65de48885f4ef01b,content_40e28f4b41764012,2026-03-01,8.395349,43,0,0,0,0,0.000000


### The trap
Adding `gsc_clicks` as a "feature" leaks the label's own numerator. Honest
score first, leaked score second, then the leaked column is dropped for good.

In [32]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

honest_features = ["gsc_avg_position", "gsc_impressions", "ga4_engaged_sessions", "sessions_ai", "scroll_events"]
feat_clean = feat.dropna(subset=honest_features + ["ctr"])

X_honest = feat_clean[honest_features]
y = feat_clean["ctr"]
r2_honest = r2_score(y, LinearRegression().fit(X_honest, y).predict(X_honest))
print(f"Honest R^2 (5 real features): {r2_honest:.3f}")

X_leaked = feat_clean[honest_features + ["gsc_clicks"]]
r2_leaked = r2_score(y, LinearRegression().fit(X_leaked, y).predict(X_leaked))
print(f"Leaked R^2 (adds gsc_clicks, the label's own numerator): {r2_leaked:.3f}")

feat_clean = feat_clean.drop(columns=["gsc_clicks"])
print(f"\nKept: honest R^2 = {r2_honest:.3f}. gsc_clicks dropped from the working frame.")

Honest R^2 (5 real features): 0.055
Leaked R^2 (adds gsc_clicks, the label's own numerator): 0.120

Kept: honest R^2 = 0.055. gsc_clicks dropped from the working frame.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**`client_has_ga4` is not stable within a client, despite reading like a fixed
attribute.** 10 of the 55 clients present in March have this flag flip between
`true` and `false` across their own rows in the same month — confirmed by
checking `COUNT(DISTINCT client_has_ga4)` per client. Any join or filter that
treats this as one constant value per client will silently miscount who has
GA4 access. This also means the earlier `ga4_data_available IS TRUE` filter is
the safer per-row check to rely on — never `client_has_ga4` alone.

**Separately, most of the panel isn't in this month at all.** Of 104 total
clients in `dim_clients`, only 55 have any March rows — 49 are simply absent,
not zero-activity. Any March-only aggregate represents about half the panel,
not the whole client base.

In [33]:
total_clients = con.sql(f"SELECT COUNT(*) AS n FROM read_parquet('{REL}/dim_clients.parquet')").df()["n"][0]
distinct_march = con.sql("SELECT COUNT(DISTINCT client_hash_id) AS n FROM march_facts").df()["n"][0]
print(f"Total clients in dim_clients: {total_clients}")
print(f"Distinct clients present in March: {distinct_march}")
print(f"Clients absent from March entirely: {total_clients - distinct_march}")

con.sql("""
    SELECT COUNT(*) AS n_clients_with_inconsistent_flag
    FROM (
        SELECT client_hash_id, COUNT(DISTINCT client_has_ga4) AS n_distinct_flags
        FROM march_facts
        GROUP BY client_hash_id
        HAVING n_distinct_flags > 1
    )
""").show()

Total clients in dim_clients: 104
Distinct clients present in March: 55
Clients absent from March entirely: 49
┌──────────────────────────────────┐
│ n_clients_with_inconsistent_flag │
│              int64               │
├──────────────────────────────────┤
│                               10 │
└──────────────────────────────────┘



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.